# Job API Scraper

List all non-deleted jobs in a Databricks workspace and return a pandas DataFrame.

Run this in a notebook within the workspace you want to query.
The SDK auto-authenticates using the notebook's context.

**Output columns:**
- `job_id`, `job_name`, `job_tags`, `task_key`, `task_type`, `cluster_type`,
  `notebook_path`, `notebook_name`, `notebook_language`,
  `last_run_duration_seconds`, `last_run_status`, `last_run_end_time`

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
CATALOG = "prod_catalog"
SCHEMA = "databricks_serverless_db"
SOURCE_TABLE = "workspace_jobs"
SUMMARY_TABLE = "workspace_jobs_summary"

full_source = f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}"
full_summary = f"{CATALOG}.{SCHEMA}.{SUMMARY_TABLE}"

In [0]:
import pandas as pd
from databricks.sdk import WorkspaceClient


def _get_cluster_type(task) -> str:
    """Determine cluster type from task compute configuration."""
    if task.job_cluster_key:
        return "job_cluster"
    if task.existing_cluster_id:
        return "interactive"
    if task.new_cluster:
        return "job_cluster"
    if getattr(task, "environment_key", None):
        return "serverless"
    if task.notebook_task or task.spark_python_task or task.sql_task:
        return "serverless"
    return "unknown"


def _get_last_run_info(client: WorkspaceClient, job_id: int) -> dict:
    """Get the most recent run's duration, status, and end time."""
    info = {
        "last_run_duration_seconds": None,
        "last_run_status": None,
        "last_run_end_time": None,
    }
    try:
        run = next(iter(client.jobs.list_runs(job_id=job_id, limit=1)), None)
        if run:
            if run.end_time and run.start_time:
                info["last_run_duration_seconds"] = round(
                    (run.end_time - run.start_time) / 1000, 1
                )
            if run.end_time:
                info["last_run_end_time"] = pd.Timestamp(
                    run.end_time, unit="ms", tz="UTC"
                )
            if run.state and run.state.result_state:
                info["last_run_status"] = run.state.result_state.value
            elif run.state and run.state.life_cycle_state:
                info["last_run_status"] = run.state.life_cycle_state.value
    except Exception:
        pass
    return info


def collect_jobs(client: WorkspaceClient) -> pd.DataFrame:
    """Iterate over all active jobs and extract task/notebook info."""
    rows = []

    print("Fetching job list...")
    all_jobs = list(client.jobs.list(expand_tasks=True))
    total_jobs = len(all_jobs)
    print(f"Found {total_jobs} jobs. Collecting details...")

    for i, job in enumerate(all_jobs, 1):
        if i % 25 == 0 or i == total_jobs:
            print(f"  Processing job {i}/{total_jobs} ({100 * i // total_jobs}%)")

        job_id = job.job_id
        job_name = job.settings.name if job.settings else None

        tags = {}
        if job.settings and job.settings.tags:
            tags = job.settings.tags
        job_tags = dict(tags) if tags else None

        run_info = _get_last_run_info(client, job_id)

        tasks = []
        if job.settings and job.settings.tasks:
            tasks = job.settings.tasks

        if not tasks:
            rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "job_tags": job_tags,
                "task_key": None,
                "task_type": None,
                "cluster_type": None,
                "notebook_path": None,
                "notebook_name": None,
                "notebook_language": None,
                **run_info,
            })
            continue

        for task in tasks:
            task_key = task.task_key
            notebook_path = None
            notebook_name = None
            notebook_language = None
            task_type = "other"

            if task.notebook_task:
                task_type = "notebook"
                notebook_path = task.notebook_task.notebook_path
                if notebook_path:
                    notebook_name = notebook_path.rsplit("/", 1)[-1]
                    try:
                        obj_info = client.workspace.get_status(notebook_path)
                        notebook_language = (
                            obj_info.language.value if obj_info.language else None
                        )
                    except Exception:
                        notebook_language = None
            elif task.spark_python_task:
                task_type = "spark_python"
            elif task.python_wheel_task:
                task_type = "python_wheel"
            elif task.spark_jar_task:
                task_type = "spark_jar"
            elif task.spark_submit_task:
                task_type = "spark_submit"
            elif task.pipeline_task:
                task_type = "pipeline"
            elif task.sql_task:
                task_type = "sql"
            elif task.dbt_task:
                task_type = "dbt"
            elif task.run_job_task:
                task_type = "run_job"

            cluster_type = _get_cluster_type(task)

            rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "job_tags": job_tags,
                "task_key": task_key,
                "task_type": task_type,
                "cluster_type": cluster_type,
                "notebook_path": notebook_path,
                "notebook_name": notebook_name,
                "notebook_language": notebook_language,
                **run_info,
            })

    df = pd.DataFrame(rows, columns=[
        "job_id", "job_name", "job_tags", "task_key", "task_type", "cluster_type",
        "notebook_path", "notebook_name", "notebook_language",
        "last_run_duration_seconds", "last_run_status", "last_run_end_time",
    ])
    print(f"Done. Collected {len(df)} task rows from {df['job_id'].nunique()} jobs.")
    return df

## Write results to Delta table

In [0]:
client = WorkspaceClient()
df = collect_jobs(client)

# ── Convert job_tags dict to string for Delta compatibility ────────────────────
df["job_tags"] = df["job_tags"].apply(lambda x: str(x) if x else None)

# ── Write to Delta table ──────────────────────────────────────────────────────
print(f"Writing {len(df)} rows to {full_source}...")

spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable(full_source)

print(f"Successfully wrote to {full_source}")

## Summarize to job-level table

In [0]:
"""                                                                                                                                                  
  Read the workspace_jobs Delta table and produce a job-level summary.                                                              
                                                                                                                                                       
  Output columns:                                                                                                                                      
    - job_id, job_name, job_tags, num_tasks, task_types,                                                                                        
      notebook_languages, cluster_types, last_run_duration_seconds,                                                                                    
      last_run_status, last_run_end_time                                                                                                               
  """

df = spark.read.table(full_source).toPandas()

summary = (
    df.groupby(["job_id", "job_name", "job_tags",
                "last_run_duration_seconds", "last_run_status", "last_run_end_time"])
    .agg(
        num_tasks=("task_key", "nunique"),
        task_types=("task_type", lambda x: dict(x.dropna().value_counts())),
        notebook_languages=("notebook_language", lambda x: dict(x.dropna().value_counts())),
        cluster_types=("cluster_type", lambda x: sorted(x.dropna().unique().tolist())),
    )
    .reset_index()
)

# Convert to readable strings
for col in ["task_types", "notebook_languages"]:
    summary[col] = summary[col].apply(
        lambda x: ", ".join(f"{k}: {v}" for k, v in x.items()) if x else None
    )
summary["cluster_types"] = summary["cluster_types"].apply(
    lambda x: ", ".join(x) if x else None
)

# ── Write to Delta table ──────────────────────────────────────────────────────
print(f"Writing {len(summary)} rows to {full_summary}...")
spark_df = spark.createDataFrame(summary)
spark_df.write.format("delta").mode("overwrite").saveAsTable(full_summary)

print(f"Successfully wrote to {full_summary}")
summary.display()